# Disaster XAI-Net — Building Localization (U-Net on pre-disaster tiles)

**Purpose.** The damage models classify *given* buildings. This notebook learns to **find the buildings** from the pre-disaster image, so the full system works on tiles without annotations:

```
 PRE tile ──► U-Net (ResNet-34 encoder) ──► building mask + border mask
                                               │ watershed on the two masks
                                               ▼
                                   individual buildings (polygons / boxes)
                                               │
 PRE + POST tile ──────────────────────────────┴──► existing Stage 2 damage model ──► severity per building
```

**Why pre-disaster only:** buildings are intact and fully visible before the event; damage (which needs the pre/post pair) is the job of the existing siamese damage model. This mirrors the two-step design of the strongest xView2 solutions.

**Why a border channel:** neighbouring buildings often touch, so a plain building mask merges them. Predicting each building's outline lets a watershed split them into individual buildings.

**Protocol (same data and split as the damage notebook):**
* development = the existing `train/` split → 90% training tiles, 10% validation tiles (model selection / early stopping only);
* the existing `test/` split is used **once**, at the end.

**Test-set evaluation:**
1. pixel-level F1 of the building mask (the xView2 localization metric);
2. building-level precision / recall / F1 (a predicted building counts as correct if IoU ≥ 0.5 with a real one);
3. **end-to-end damage assessment**: predicted buildings are passed through the *existing final Stage 2 damage model* and compared with the same model run on the ground-truth footprints (needs the damage notebook's `deployment_bundle` attached as an input).

Kaggle: **GPU on, Internet on**. Input: the same `hazard_data` dataset as the damage notebook; optional: the damage notebook's output (for section 8).

In [ ]:
import os, re, gc, json, time, math, random, glob, shutil, zipfile, datetime, sys, platform, importlib.util
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
import timm
from scipy import ndimage as ndi
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, cohen_kappa_score, matthews_corrcoef, confusion_matrix
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
try:
    from skimage.segmentation import watershed
    from skimage.measure import find_contours, approximate_polygon
    HAVE_SKIMAGE = True
except Exception:
    HAVE_SKIMAGE = False
Image.MAX_IMAGE_PIXELS = None

_env = os.environ.get
SMOKE = _env("LOC_SMOKE", "0") == "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = torch.cuda.is_available()
print("device:", device, "| skimage:", HAVE_SKIMAGE, "| timm", timm.__version__)

ENCODER     = "resnet34"
CROP        = int(_env("LOC_CROP", 512))       # random training crops from the 1024 px tiles
BATCH       = int(_env("LOC_BATCH", 8))
EPOCHS      = int(_env("LOC_EPOCHS", 30))
PATIENCE    = 6
LR, WD      = 3e-4, 1e-4
BORDER_PX   = 3                                 # outline width drawn for the border target
BORDER_W    = 0.5                               # loss weight of the border channel
THR_MASK, THR_BORDER = 0.5, 0.5                 # fixed, not tuned on test
MIN_AREA    = int(_env("LOC_MIN_AREA", 20))     # px; smaller predicted blobs are discarded
IOU_MATCH   = 0.5
VAL_FRAC, SEED = 0.10, 42
NUM_WORKERS = int(_env("LOC_NUM_WORKERS", 2))
RESUME, FORCE_RETRAIN = True, False

OUT = _env("LOC_OUTPUT_ROOT", "/kaggle/working/localization_results")
D = {k: os.path.join(OUT, v) for k, v in {'ckpt': 'checkpoints', 'metrics': 'metrics', 'plots': 'plots',
     'pred': 'predictions', 'bundle': 'localization_bundle', 'geo': 'geojson', 'repro': 'reproducibility'}.items()}
for p in D.values(): os.makedirs(p, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def jdump(o, p):
    t = p + ".tmp"
    with open(t, "w") as fh: json.dump(o, fh, indent=2, default=lambda x: x.item() if hasattr(x, 'item') else str(x))
    os.replace(t, p)

def atomic_save(o, p):
    t = p + ".tmp"; torch.save(o, t); os.replace(t, p)

HYPER = dict(encoder=ENCODER, crop=CROP, batch=BATCH, epochs=EPOCHS, patience=PATIENCE, lr=LR, wd=WD,
             border_px=BORDER_PX, border_w=BORDER_W, thr_mask=THR_MASK, thr_border=THR_BORDER, min_area=MIN_AREA,
             iou_match=IOU_MATCH, val_frac=VAL_FRAC, seed=SEED, amp=AMP, selection='val pixel F1')
jdump(HYPER, os.path.join(D['repro'], "hyperparameters.json"))
set_seed(SEED)

---
## 1. Data — same tiles and split as the damage notebook

In [ ]:
hazard_base_path = _env("LOC_DATA_ROOT")
if not hazard_base_path:
    for b in ["/kaggle/input", "/content"]:
        if os.path.isdir(b):
            for dp, _, _ in os.walk(b):
                if os.path.basename(dp) == "hazard_data":
                    hazard_base_path = dp; break
        if hazard_base_path: break
assert hazard_base_path, "hazard_data not found - add the same dataset input as the damage notebook"
print("hazard_base_path:", hazard_base_path)

def prefix_of(fn):
    m = re.match(r'^(.*?)_(pre|post)_disaster\.png$', fn)
    return m.group(1) if m else None

records = []
for hazard in sorted(os.listdir(hazard_base_path)):
    for split in ['train', 'test']:
        img_dir = os.path.join(hazard_base_path, hazard, split, 'images')
        lbl_dir = os.path.join(hazard_base_path, hazard, split, 'labels')
        if not os.path.isdir(img_dir): continue
        files = set(os.listdir(img_dir))
        for p in sorted({prefix_of(f) for f in files if prefix_of(f)}):
            pre, post = f"{p}_pre_disaster.png", f"{p}_post_disaster.png"
            if pre in files and post in files:
                pre_lbl = os.path.join(lbl_dir, f"{p}_pre_disaster.json")
                post_lbl = os.path.join(lbl_dir, f"{p}_post_disaster.json")
                records.append({'hazard': hazard, 'split': split, 'tile': p,
                                'pre_path': os.path.join(img_dir, pre), 'post_path': os.path.join(img_dir, post),
                                'pre_label': pre_lbl if os.path.exists(pre_lbl) else post_lbl,
                                'post_label': post_lbl})
tiles = pd.DataFrame(records).sort_values('tile').reset_index(drop=True)
assert not tiles['tile'].duplicated().any()
dev = tiles[tiles['split'] == 'train'].reset_index(drop=True)
test = tiles[tiles['split'] == 'test'].reset_index(drop=True)
tr_idx, va_idx = train_test_split(np.arange(len(dev)), test_size=VAL_FRAC, random_state=SEED, stratify=dev['hazard'])
train_t, val_t = dev.iloc[np.sort(tr_idx)].reset_index(drop=True), dev.iloc[np.sort(va_idx)].reset_index(drop=True)
assert not (set(train_t['tile']) & set(val_t['tile'])) and not (set(dev['tile']) & set(test['tile']))
print(f"train {len(train_t)} | val {len(val_t)} | test {len(test)} tiles  (test untouched until section 7)")
print(pd.crosstab(tiles['hazard'], tiles['split']))
print("pre-disaster label files found for", f"{(tiles['pre_label'].str.contains('_pre_')).mean():.0%}", "of tiles (else post-label polygons are used)")
pd.concat([train_t.assign(role='train'), val_t.assign(role='val'), test.assign(role='test')])[['tile', 'hazard', 'role']].to_csv(
    os.path.join(D['repro'], "tile_split.csv"), index=False)

### Polygons → targets

`building` = filled footprints; `border` = each footprint's outline (width 3 px). An instance map (one id per building) is also rasterised for the building-level evaluation.

In [ ]:
WKT_RE = re.compile(r'\(\((.*?)\)\)')

def parse_wkt(wkt):
    m = WKT_RE.search(wkt or '')
    if not m: return None
    try:
        pts = [tuple(float(v) for v in c.strip().split()[:2]) for c in m.group(1).split(',')]
    except ValueError:
        return None
    return pts if len(pts) >= 3 else None

def read_buildings(label_path, with_damage=False):
    """list of dicts {uid, xy:[(x,y)...], lnglat:[(lon,lat)...]|None, subtype}"""
    js = json.load(open(label_path))
    feats = js.get('features', {})
    ll = {}
    for f in feats.get('lng_lat', []) or []:
        uid = f.get('properties', {}).get('uid'); pts = parse_wkt(f.get('wkt'))
        if uid and pts: ll[uid] = pts
    out = []
    for f in feats.get('xy', []) or []:
        pts = parse_wkt(f.get('wkt'))
        if not pts: continue
        pr = f.get('properties', {})
        out.append({'uid': pr.get('uid'), 'xy': pts, 'lnglat': ll.get(pr.get('uid')),
                    'subtype': pr.get('subtype', 'un-classified')})
    return out

def rasterize(buildings, size, instances=False):
    W, H = size
    mask = Image.new('L', (W, H), 0); border = Image.new('L', (W, H), 0)
    dm, db = ImageDraw.Draw(mask), ImageDraw.Draw(border)
    inst = Image.new('I', (W, H), 0) if instances else None
    di = ImageDraw.Draw(inst) if instances else None
    for k, b in enumerate(buildings, 1):
        dm.polygon(b['xy'], fill=1)
        db.line(b['xy'] + [b['xy'][0]], fill=1, width=BORDER_PX)
        if instances: di.polygon(b['xy'], fill=k)
    m, bo = np.asarray(mask, np.uint8), np.asarray(border, np.uint8)
    return (m, bo, np.asarray(inst, np.int32)) if instances else (m, bo)

_b = read_buildings(train_t['pre_label'].iloc[0])
print(f"example tile: {len(_b)} buildings; tile size {Image.open(train_t['pre_path'].iloc[0]).size}")

---
## 2. Dataset and augmentation

Training: random `CROP`×`CROP` crops (biased so ~2/3 of crops contain buildings), random 90° rotations and flips, mild brightness/contrast jitter. Validation/test: full tiles, no augmentation.

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)

def to_tensor(img_u8):
    return torch.from_numpy(((img_u8.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1).copy())

TARGET_CACHE = {}
def tile_targets(row):
    if row['tile'] not in TARGET_CACHE:
        size = Image.open(row['pre_path']).size
        TARGET_CACHE[row['tile']] = rasterize(read_buildings(row['pre_label']), size)
    return TARGET_CACHE[row['tile']]

class TrainDS(Dataset):
    def __init__(self, df, crops_per_tile=2):
        self.df = df.reset_index(drop=True); self.n = crops_per_tile
    def __len__(self): return len(self.df) * self.n
    def __getitem__(self, i):
        row = self.df.iloc[i % len(self.df)]
        img = np.asarray(Image.open(row['pre_path']).convert('RGB'))
        m, b = rasterize(read_buildings(row['pre_label']), (img.shape[1], img.shape[0]))
        H, W = m.shape; c = min(CROP, H, W)
        for _ in range(10):                                   # prefer crops with buildings
            y0, x0 = random.randint(0, H - c), random.randint(0, W - c)
            if m[y0:y0 + c, x0:x0 + c].any() or random.random() < 0.33: break
        img, m, b = img[y0:y0 + c, x0:x0 + c], m[y0:y0 + c, x0:x0 + c], b[y0:y0 + c, x0:x0 + c]
        k = random.randint(0, 3)
        img, m, b = np.rot90(img, k), np.rot90(m, k), np.rot90(b, k)
        if random.random() < 0.5: img, m, b = img[:, ::-1], m[:, ::-1], b[:, ::-1]
        if random.random() < 0.5: img, m, b = img[::-1], m[::-1], b[::-1]
        img = img.astype(np.float32)
        img = np.clip((img - 128) * random.uniform(0.85, 1.15) + 128 + random.uniform(-15, 15), 0, 255).astype(np.uint8)
        y = torch.from_numpy(np.stack([m, b]).astype(np.float32).copy())
        return to_tensor(img), y

class TileDS(Dataset):
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = np.asarray(Image.open(row['pre_path']).convert('RGB'))
        m, b = tile_targets(row)
        return to_tensor(img), torch.from_numpy(np.stack([m, b]).astype(np.float32)), i

---
## 3. Model — U-Net decoder on a pretrained ResNet-34 encoder

Encoder: `timm` ResNet-34 (ImageNet), multi-scale features at strides 2–32. Decoder: standard U-Net upsampling blocks with skip connections. Two output channels: building and border.

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(cin + cskip, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                                  nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        if skip is not None: x = torch.cat([x, skip], 1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, encoder=ENCODER, pretrained=True, n_out=2, dec=(256, 128, 64, 32, 16)):
        super().__init__()
        self.enc = timm.create_model(encoder, features_only=True, pretrained=pretrained)
        ch = self.enc.feature_info.channels()                 # e.g. [64, 64, 128, 256, 512]
        skips = ch[:-1][::-1] + [0]
        cins = [ch[-1]] + list(dec[:-1])
        self.blocks = nn.ModuleList([DecoderBlock(ci, cs, co) for ci, cs, co in zip(cins, skips, dec)])
        self.head = nn.Conv2d(dec[-1], n_out, 1)
    def forward(self, x):
        f = self.enc(x)
        skips = f[:-1][::-1] + [None]
        y = f[-1]
        for blk, s in zip(self.blocks, skips):
            y = blk(y, s)
        return self.head(y)

def loss_fn(logits, y):
    bce = F.binary_cross_entropy_with_logits(logits, y, reduction='none').mean((0, 2, 3))
    p = torch.sigmoid(logits[:, 0]); t = y[:, 0]
    dice = 1 - (2 * (p * t).sum() + 1) / (p.sum() + t.sum() + 1)
    return bce[0] + dice + BORDER_W * bce[1]

_m = UNet(pretrained=False)
with torch.no_grad():
    assert _m(torch.zeros(1, 3, 64, 64)).shape == (1, 2, 64, 64)
print(f"U-Net params: {sum(p.numel() for p in _m.parameters())/1e6:.1f} M"); del _m

---
## 4. Post-processing — masks → individual buildings

Building pixels = P(building) > 0.5. Seeds = building pixels that are not border (P(border) < 0.5), split into connected components. A watershed grows each seed back over the full building mask, so touching buildings come out as separate instances. Blobs smaller than `MIN_AREA` px are dropped. Thresholds are fixed in advance, not tuned on test.

In [ ]:
def instances_from_probs(pm, pb):
    fg = pm > THR_MASK
    seeds, n = ndi.label(fg & (pb < THR_BORDER))
    if n:
        sizes = ndi.sum(np.ones_like(seeds), seeds, index=np.arange(1, n + 1))
        seeds[np.isin(seeds, np.where(sizes < 4)[0] + 1)] = 0
    if HAVE_SKIMAGE and seeds.any():
        inst = watershed(-pm, seeds, mask=fg)
    else:
        inst, _ = ndi.label(fg)
    ids, cnt = np.unique(inst[inst > 0], return_counts=True)
    keep = ids[cnt >= MIN_AREA]
    remap = np.zeros(inst.max() + 1, np.int32); remap[keep] = np.arange(1, len(keep) + 1)
    return remap[inst]

def instance_polygons(inst):
    """instance map -> list of {id, bbox [x0,y0,x1,y1], area, polygon [[x,y],...]} in image pixels"""
    out = []
    for k, sl in enumerate(ndi.find_objects(inst), 1):
        if sl is None: continue
        ys, xs = sl
        sub = (inst[sl] == k)
        poly = None
        if HAVE_SKIMAGE:
            cs = find_contours(np.pad(sub, 1).astype(float), 0.5)
            if cs:
                c = approximate_polygon(max(cs, key=len), tolerance=1.0)
                poly = [[float(x - 1 + xs.start), float(y - 1 + ys.start)] for y, x in c]
        if not poly or len(poly) < 3:
            poly = [[xs.start, ys.start], [xs.stop, ys.start], [xs.stop, ys.stop], [xs.start, ys.stop], [xs.start, ys.start]]
        out.append({'id': k, 'bbox': [float(xs.start), float(ys.start), float(xs.stop), float(ys.stop)],
                    'area': int(sub.sum()), 'polygon': poly})
    return out

def match_instances(gt_inst, pr_inst):
    """greedy one-to-one matching at IoU >= IOU_MATCH (unique at 0.5). Returns list of (gt_id, pred_id, iou)."""
    ng, npd = gt_inst.max(), pr_inst.max()
    if ng == 0 or npd == 0: return []
    both = (gt_inst > 0) & (pr_inst > 0)
    inter = np.bincount(gt_inst[both].astype(np.int64) * (npd + 1) + pr_inst[both], minlength=(ng + 1) * (npd + 1)).reshape(ng + 1, npd + 1)
    ag, ap = np.bincount(gt_inst.ravel(), minlength=ng + 1), np.bincount(pr_inst.ravel(), minlength=npd + 1)
    g, p = np.nonzero(inter)
    iou = inter[g, p] / (ag[g] + ap[p] - inter[g, p])
    order = np.argsort(-iou); used_g, used_p, res = set(), set(), []
    for o in order:
        if iou[o] < IOU_MATCH: break
        if g[o] in used_g or p[o] in used_p: continue
        used_g.add(g[o]); used_p.add(p[o]); res.append((int(g[o]), int(p[o]), float(iou[o])))
    return res
print("post-processing ready")

---
## 5. Training (resumable per epoch; selection on validation pixel F1)

In [ ]:
@torch.no_grad()
def predict_probs(model, img_t):
    """img_t: (B,3,H,W) normalised; pads to a multiple of 32; returns sigmoid probs (B,2,H,W) on CPU"""
    model.eval()
    H, W = img_t.shape[-2:]
    ph, pw = (32 - H % 32) % 32, (32 - W % 32) % 32
    x = F.pad(img_t, (0, pw, 0, ph), mode='reflect') if (ph or pw) else img_t
    with torch.amp.autocast('cuda', enabled=AMP):
        p = torch.sigmoid(model(x.to(device)).float())
    return p[..., :H, :W].cpu()

def pixel_counts(model, loader):
    tp = fp = fn = 0
    for x, y, _ in loader:
        p = predict_probs(model, x)[:, 0] > THR_MASK
        t = y[:, 0] > 0.5
        tp += int((p & t).sum()); fp += int((p & ~t).sum()); fn += int((~p & t).sum())
    return tp, fp, fn

def f1_from(tp, fp, fn): return 2 * tp / max(2 * tp + fp + fn, 1)

CK = D['ckpt']; last_p, best_p, done_p = (os.path.join(CK, f) for f in ("last.pth", "best.pth", "done.json"))
if FORCE_RETRAIN:
    for p in (last_p, best_p, done_p):
        if os.path.exists(p): os.remove(p)

if RESUME and os.path.exists(done_p):
    TRAIN_INFO = json.load(open(done_p)); print("[RESUME] training already complete:", {k: TRAIN_INFO[k] for k in ('best_epoch', 'best_val_pixel_f1')})
else:
    set_seed(SEED)
    model = UNet(pretrained=not SMOKE).to(device)
    enc_params = list(model.enc.parameters()); enc_ids = {id(p) for p in enc_params}
    opt = torch.optim.AdamW([{'params': enc_params, 'lr': LR / 3},
                             {'params': [p for p in model.parameters() if id(p) not in enc_ids], 'lr': LR}], weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    tl = DataLoader(TrainDS(train_t), batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    vl = DataLoader(TileDS(val_t), batch_size=2, num_workers=NUM_WORKERS)
    hist, best_f1, best_ep, bad, start = [], -1.0, 0, 0, 1
    if RESUME and os.path.exists(last_p):
        ck = torch.load(last_p, map_location='cpu', weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch']); scaler.load_state_dict(ck['scaler'])
        hist, best_f1, best_ep, bad, start = ck['hist'], ck['best_f1'], ck['best_ep'], ck['bad'], ck['epoch'] + 1
        print(f"[RESUME] continuing at epoch {start}"); del ck
    for ep in range(start, EPOCHS + 1):
        if bad >= PATIENCE: break
        model.train(); t0 = time.time(); tot = n = 0
        for x, y in tqdm(tl, desc=f"epoch {ep}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=AMP):
                loss = loss_fn(model(x).float(), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(x); n += len(x)
        sch.step()
        vf = f1_from(*pixel_counts(model, vl))
        hist.append({'epoch': ep, 'train_loss': tot / max(n, 1), 'val_pixel_f1': vf})
        mark = ""
        if vf > best_f1:
            best_f1, best_ep, bad = vf, ep, 0
            atomic_save({'state_dict': {k: v.cpu() for k, v in model.state_dict().items()},
                         'meta': {'encoder': ENCODER, 'epoch': ep, 'val_pixel_f1': vf, 'thr_mask': THR_MASK,
                                  'thr_border': THR_BORDER, 'min_area': MIN_AREA, 'mean': MEAN.tolist(), 'std': STD.tolist()}}, best_p)
            mark = "  <- best (saved)"
        else:
            bad += 1
        atomic_save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sch': sch.state_dict(), 'scaler': scaler.state_dict(),
                     'epoch': ep, 'hist': hist, 'best_f1': best_f1, 'best_ep': best_ep, 'bad': bad}, last_p)
        print(f"epoch {ep:2d}/{EPOCHS} | train loss {tot/max(n,1):.4f} | val pixel F1 {vf:.4f} | {time.time()-t0:.0f}s{mark}")
    TRAIN_INFO = {'best_epoch': best_ep, 'best_val_pixel_f1': best_f1, 'epochs_run': len(hist), 'history': hist}
    jdump(TRAIN_INFO, done_p)
    if os.path.exists(last_p): os.remove(last_p)
    del model, opt; gc.collect(); torch.cuda.empty_cache()

h = pd.DataFrame(TRAIN_INFO['history']); h.to_csv(os.path.join(D['metrics'], "training_history.csv"), index=False)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h['epoch'], h['train_loss'], marker='o'); ax[0].set_title('train loss')
ax[1].plot(h['epoch'], h['val_pixel_f1'], marker='o'); ax[1].set_title('validation pixel F1')
for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3); a.axvline(TRAIN_INFO['best_epoch'], ls='--', c='k', lw=1)
plt.tight_layout(); plt.savefig(os.path.join(D['plots'], "training_curves.png"), dpi=150); plt.show(); plt.close(fig)

---
## 6. Load the selected model

In [ ]:
model = UNet(pretrained=False).to(device)
_ck = torch.load(best_p, map_location='cpu', weights_only=False)
model.load_state_dict(_ck['state_dict']); model.eval()
print("selected model:", _ck['meta'])

---
## 7. FINAL HELD-OUT TEST — localization (once)

Per test tile: pixel confusion counts, building-level matches (IoU ≥ 0.5), and the predicted buildings themselves (saved for section 8 and the demo).

In [ ]:
TEST_CACHE = os.path.join(D['pred'], "test_instances.json")
pix = {'tp': 0, 'fp': 0, 'fn': 0}; obj = {'tp': 0, 'n_gt': 0, 'n_pred': 0}
per_tile, PRED = [], {}
for i in tqdm(range(len(test)), desc="test tiles"):
    row = test.iloc[i]
    img = np.asarray(Image.open(row['pre_path']).convert('RGB'))
    gt_b = read_buildings(row['pre_label'])
    gm, _, gi = rasterize(gt_b, (img.shape[1], img.shape[0]), instances=True)
    p = predict_probs(model, to_tensor(img)[None])[0].numpy()
    pm = p[0] > THR_MASK; t = gm > 0
    tp, fp, fn = int((pm & t).sum()), int((pm & ~t).sum()), int((~pm & t).sum())
    inst = instances_from_probs(p[0], p[1])
    m = match_instances(gi, inst)
    pix['tp'] += tp; pix['fp'] += fp; pix['fn'] += fn
    obj['tp'] += len(m); obj['n_gt'] += int(gi.max()); obj['n_pred'] += int(inst.max())
    polys = instance_polygons(inst)
    PRED[row['tile']] = {'buildings': polys, 'matches': m, 'size': [img.shape[1], img.shape[0]]}
    per_tile.append({'tile': row['tile'], 'hazard': row['hazard'], 'n_gt': int(gi.max()), 'n_pred': int(inst.max()),
                     'matched': len(m), 'pixel_f1': f1_from(tp, fp, fn) if (tp + fp + fn) else np.nan})
jdump(PRED, TEST_CACHE)
pt = pd.DataFrame(per_tile); pt.to_csv(os.path.join(D['metrics'], "test_per_tile.csv"), index=False)

prec = obj['tp'] / max(obj['n_pred'], 1); rec = obj['tp'] / max(obj['n_gt'], 1)
LOC_TEST = {'pixel_precision': pix['tp'] / max(pix['tp'] + pix['fp'], 1), 'pixel_recall': pix['tp'] / max(pix['tp'] + pix['fn'], 1),
            'pixel_f1': f1_from(pix['tp'], pix['fp'], pix['fn']),
            'building_precision': prec, 'building_recall': rec, 'building_f1': 2 * prec * rec / max(prec + rec, 1e-9),
            'n_test_tiles': len(test), 'n_gt_buildings': obj['n_gt'], 'n_pred_buildings': obj['n_pred'], 'iou_threshold': IOU_MATCH}
by_h = []
for hz, g in pt.groupby('hazard'):
    pr_, rc_ = g['matched'].sum() / max(g['n_pred'].sum(), 1), g['matched'].sum() / max(g['n_gt'].sum(), 1)
    by_h.append({'hazard': hz, 'tiles': len(g), 'gt_buildings': int(g['n_gt'].sum()), 'building_precision': pr_,
                 'building_recall': rc_, 'building_f1': 2 * pr_ * rc_ / max(pr_ + rc_, 1e-9)})
by_h = pd.DataFrame(by_h).round(4); by_h.to_csv(os.path.join(D['metrics'], "test_building_f1_by_hazard.csv"), index=False)
jdump(LOC_TEST, os.path.join(D['metrics'], "test_localization_metrics.json"))
print("=" * 64); print("FINAL HELD-OUT TEST - LOCALIZATION"); print("=" * 64)
print(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in LOC_TEST.items()}, indent=1))
print(by_h.to_string(index=False))

# qualitative figure: best, median and worst tiles by pixel F1 (among tiles with buildings)
q = pt[pt['n_gt'] > 0].sort_values('pixel_f1')
pick = [q.iloc[-1], q.iloc[len(q) // 2], q.iloc[0]] if len(q) >= 3 else [q.iloc[0]]
fig, axes = plt.subplots(len(pick), 3, figsize=(12, 4 * len(pick)))
axes = np.atleast_2d(axes)
for r, (row, tag) in enumerate(zip(pick, ['best', 'median', 'worst'])):
    trow = test[test['tile'] == row['tile']].iloc[0]
    img = np.asarray(Image.open(trow['pre_path']).convert('RGB'))
    gm, _ = rasterize(read_buildings(trow['pre_label']), (img.shape[1], img.shape[0]))
    p = predict_probs(model, to_tensor(img)[None])[0].numpy()
    inst = instances_from_probs(p[0], p[1])
    axes[r, 0].imshow(img); axes[r, 0].set_title(f"{tag}: {row['tile']} (pixel F1 {row['pixel_f1']:.2f})", fontsize=9)
    axes[r, 1].imshow(img); axes[r, 1].imshow(np.ma.masked_equal(gm, 0), alpha=.5, cmap='autumn'); axes[r, 1].set_title('ground-truth footprints', fontsize=9)
    axes[r, 2].imshow(img); axes[r, 2].imshow(np.ma.masked_equal(inst, 0) % 20, alpha=.6, cmap='tab20'); axes[r, 2].set_title(f'predicted buildings ({inst.max()})', fontsize=9)
    for a in axes[r]: a.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(D['plots'], "test_examples.png"), dpi=120, bbox_inches='tight'); plt.show(); plt.close(fig)

---
## 8. End-to-end damage assessment with predicted footprints

Requires the damage notebook's output attached as an input (`disaster_xai_net_master_results/deployment_bundle/inference.py`). The **same final Stage 2 model** is run twice on every test tile:

* **oracle**: on the ground-truth footprints (should reproduce the damage notebook's natural-distribution test numbers — a sanity check);
* **predicted**: on the U-Net's buildings.

Each ground-truth building (un-classified dropped) takes the damage label of its matched predicted building. Reported:
* detection recall per true damage class (are destroyed buildings found as often as intact ones?);
* severity metrics on the **matched** buildings for oracle vs predicted footprints (isolates the effect of footprint quality);
* **end-to-end** metrics over all ground-truth buildings, where a missed building counts as predicted `no-damage` (a conservative, stated convention);
* predicted buildings that match nothing (false positives) and what damage they were given.

In [ ]:
DAMAGE = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
bundle_py = None
for root in [_env("LOC_DAMAGE_BUNDLE_ROOT"), "/kaggle/input", "/kaggle/working"]:
    if root and os.path.isdir(root):
        hits = sorted(glob.glob(os.path.join(root, "**", "deployment_bundle", "inference.py"), recursive=True))
        if hits:
            print("damage bundles found:\n  " + "\n  ".join(hits))
            # the archive may also hold the single-backbone runs - always prefer the fusion ensemble
            fusion = [h for h in hits if 'disaster_xai_net_master_results' in h]
            if not fusion:
                for h in hits:
                    try:
                        cfg_ = json.load(open(os.path.join(os.path.dirname(h), 'model_config.json')))
                        if 'fusion' in str(cfg_.get('arch_tag', '')).lower() or 'Fusion' in str(cfg_.get('model_name', '')):
                            fusion.append(h)
                    except Exception:
                        pass
            bundle_py = (fusion or hits)[0]
            break
if bundle_py:
    _ck_dir = os.path.join(os.path.dirname(bundle_py), 'checkpoints')
    print("USING damage bundle:", bundle_py)
    print("  checkpoints present:", sorted(os.listdir(_ck_dir)) if os.path.isdir(_ck_dir) else "NONE - re-upload the bundle with its .pth files")
E2E = None
if bundle_py is None:
    print("damage deployment_bundle not attached - end-to-end evaluation skipped. Attach the damage notebook's output and re-run.")
else:
    del model; gc.collect(); torch.cuda.empty_cache()
    spec = importlib.util.spec_from_file_location("dxn_damage", bundle_py)
    dmg = importlib.util.module_from_spec(spec); spec.loader.exec_module(dmg)
    dmg.load_models(os.path.dirname(bundle_py))
    rows, fps, sev_rows = [], [], []
    for i in tqdm(range(len(test)), desc="end-to-end damage"):
        row = test.iloc[i]
        post_b = read_buildings(row['post_label'])
        pre_b = read_buildings(row['pre_label'])
        # ground-truth instance ids follow the PRE label order (the matching above); map pre uid -> post subtype
        sub_by_uid = {b['uid']: b['subtype'] for b in post_b}
        P = PRED[row['tile']]
        if not pre_b: continue
        gt_req = [{'building_id': f"g{k}", 'wkt': 'POLYGON ((' + ', '.join(f"{x} {y}" for x, y in b['xy']) + '))'}
                  for k, b in enumerate(pre_b, 1)]
        oracle = {r['building_id']: r for r in dmg.predict_full(row['pre_path'], row['post_path'], buildings=gt_req)['buildings']}
        pred_req = [{'building_id': f"p{b['id']}", 'bbox': b['bbox']} for b in P['buildings']]
        predr = {r['building_id']: r for r in dmg.predict_full(row['pre_path'], row['post_path'], buildings=pred_req)['buildings']} if pred_req else {}
        for b in P['buildings']:
            r_ = predr[f"p{b['id']}"]
            sev_rows.append({'tile': row['tile'], 'pred_building_id': b['id'], 'severity': r_['severity'],
                             'p_damaged': r_['p_damaged'], 'confidence': r_['confidence']})
        g2p = {g: p for g, p, _ in P['matches']}
        for k, b in enumerate(pre_b, 1):
            sub = sub_by_uid.get(b['uid'], b['subtype'])
            if sub not in DAMAGE: continue
            pm = g2p.get(k)
            rows.append({'tile': row['tile'], 'hazard': row['hazard'], 'gt': sub, 'oracle': oracle[f"g{k}"]['severity'],
                         'matched': pm is not None, 'pred': predr[f"p{pm}"]['severity'] if pm is not None else None})
        matched_p = set(g2p.values())
        for b in P['buildings']:
            if b['id'] not in matched_p:
                fps.append({'tile': row['tile'], 'severity': predr[f"p{b['id']}"]['severity'], 'area': b['area']})
    e = pd.DataFrame(rows); fpd = pd.DataFrame(fps)
    e.to_csv(os.path.join(D['pred'], "end_to_end_buildings.csv"), index=False)
    fpd.to_csv(os.path.join(D['pred'], "end_to_end_false_positives.csv"), index=False)
    pd.DataFrame(sev_rows).to_csv(os.path.join(D['pred'], "end_to_end_predicted_building_severity.csv"), index=False)
    idx = {n: i for i, n in enumerate(DAMAGE)}

    def sev_metrics(y, p):
        y, p = [idx[v] for v in y], [idx[v] for v in p]
        return {'n': len(y), 'macro_f1': f1_score(y, p, average='macro', labels=range(4), zero_division=0),
                'accuracy': accuracy_score(y, p), 'qwk': cohen_kappa_score(y, p, weights='quadratic'),
                'mcc': matthews_corrcoef(y, p)}
    mt = e[e['matched']]
    E2E = {
        'oracle_all_gt_buildings': sev_metrics(e['gt'], e['oracle']),
        'oracle_on_matched': sev_metrics(mt['gt'], mt['oracle']),
        'predicted_footprints_on_matched': sev_metrics(mt['gt'], mt['pred']),
        'end_to_end_missed_as_no_damage': sev_metrics(e['gt'], e['pred'].fillna('no-damage')),
        'detection_recall_by_true_class': e.groupby('gt')['matched'].mean().reindex(DAMAGE).round(4).to_dict(),
        'n_false_positive_buildings': len(fpd),
        'false_positive_severity_distribution': fpd['severity'].value_counts().to_dict() if len(fpd) else {},
        'convention': 'missed ground-truth buildings are counted as predicted no-damage',
    }
    jdump(E2E, os.path.join(D['metrics'], "test_end_to_end_damage.json"))
    tab = pd.DataFrame({k: v for k, v in E2E.items() if isinstance(v, dict) and 'macro_f1' in v}).T.round(4)
    tab.to_csv(os.path.join(D['metrics'], "test_end_to_end_damage_table.csv"))
    print("=" * 64); print("FINAL HELD-OUT TEST - END-TO-END DAMAGE"); print("=" * 64)
    print(tab.to_string()); print("\ndetection recall by true damage class:", E2E['detection_recall_by_true_class'])
    print("false-positive buildings:", E2E['n_false_positive_buildings'], E2E['false_positive_severity_distribution'])
    cm = confusion_matrix([idx[v] for v in e['gt']], [idx[v] for v in e['pred'].fillna('no-damage')], labels=range(4))
    pd.DataFrame(cm, index=DAMAGE, columns=DAMAGE).to_csv(os.path.join(D['metrics'], "end_to_end_confusion_matrix.csv"))
    dmg._STATE.clear(); del dmg; gc.collect(); torch.cuda.empty_cache()

---
## 9. GeoJSON for the demo map + localization bundle

xBD tiles are distributed as PNGs without geo-referencing, but each label file lists every building in both pixel and longitude/latitude coordinates. For each test tile an affine pixel→(lon, lat) transform is fitted by least squares on those vertex pairs and applied to the **predicted** polygons. In a real deployment this transform comes from the GeoTIFF metadata of the imagery instead. Severity is attached when section 8 ran.

In [ ]:
def fit_affine(buildings):
    src, dst = [], []
    for b in buildings:
        if b['lnglat'] and len(b['lnglat']) == len(b['xy']):
            src += b['xy']; dst += b['lnglat']
    if len(src) < 3: return None
    A = np.c_[np.asarray(src), np.ones(len(src))]
    coef, *_ = np.linalg.lstsq(A, np.asarray(dst), rcond=None)
    resid = np.abs(A @ coef - np.asarray(dst)).max()
    return coef if np.linalg.matrix_rank(A) == 3 else None, float(resid)

PRED_SEV = {}
_ps = os.path.join(D['pred'], "end_to_end_predicted_building_severity.csv")
if E2E is not None and os.path.exists(_ps):
    for r in pd.read_csv(_ps).itertuples():
        PRED_SEV[(r.tile, int(r.pred_building_id))] = {'severity': r.severity, 'p_damaged': float(r.p_damaged),
                                                       'severity_confidence': float(r.confidence)}
features, georef = [], []
for row in test.itertuples():
    P = PRED[row.tile]
    fa = fit_affine(read_buildings(row.post_label))
    if fa is None or fa[0] is None: continue
    coef, resid = fa; georef.append({'tile': row.tile, 'max_residual_deg': resid})
    for b in P['buildings']:
        xy = np.c_[np.asarray(b['polygon']), np.ones(len(b['polygon']))] @ coef
        ring = xy.tolist()
        if ring[0] != ring[-1]: ring.append(ring[0])
        features.append({'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [ring]},
                         'properties': {'tile': row.tile, 'hazard': row.hazard, 'pred_building_id': b['id'],
                                        'area_px': b['area'], 'source': 'U-Net prediction (pre-disaster image)',
                                        **PRED_SEV.get((row.tile, b['id']), {})}})
jdump({'type': 'FeatureCollection', 'features': features}, os.path.join(D['geo'], "predicted_buildings_test.geojson"))
pd.DataFrame(georef).to_csv(os.path.join(D['geo'], "georeference_residuals.csv"), index=False)
print(f"GeoJSON: {len(features)} predicted buildings on {len(georef)} geo-referenced test tiles")

# ---- localization bundle: predict_buildings(pre_image) ----
B = D['bundle']; os.makedirs(B, exist_ok=True)
shutil.copy2(best_p, os.path.join(B, "unet_localization.pth"))
LOC_PY = r'''"""
Building localization from a PRE-disaster tile.

    from localization_inference import load, predict_buildings
    load()
    buildings = predict_buildings("tile_pre_disaster.png")
    # -> [{'id', 'bbox':[x0,y0,x1,y1], 'area', 'polygon':[[x,y],...]}, ...] in image pixels
    # each item can be passed directly to the damage bundle: predict_full(pre, post, buildings=[{'bbox': b['bbox']}, ...])
"""
import os, io
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
from PIL import Image
from scipy import ndimage as ndi
try:
    from skimage.segmentation import watershed
    from skimage.measure import find_contours, approximate_polygon
    _SK = True
except Exception:
    _SK = False

_S = {}
class DecoderBlock(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(cin + cskip, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                                  nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        if skip is not None: x = torch.cat([x, skip], 1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, encoder, n_out=2, dec=(256, 128, 64, 32, 16)):
        super().__init__()
        self.enc = timm.create_model(encoder, features_only=True, pretrained=False)
        ch = self.enc.feature_info.channels(); skips = ch[:-1][::-1] + [0]; cins = [ch[-1]] + list(dec[:-1])
        self.blocks = nn.ModuleList([DecoderBlock(ci, cs, co) for ci, cs, co in zip(cins, skips, dec)])
        self.head = nn.Conv2d(dec[-1], n_out, 1)
    def forward(self, x):
        f = self.enc(x); skips = f[:-1][::-1] + [None]; y = f[-1]
        for blk, s in zip(self.blocks, skips): y = blk(y, s)
        return self.head(y)

def load(path=None, device=None):
    if 'model' in _S: return _S
    path = path or os.path.join(os.path.dirname(os.path.abspath(__file__)), 'unet_localization.pth')
    ck = torch.load(path, map_location='cpu', weights_only=False)
    dev = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    m = UNet(ck['meta']['encoder']); m.load_state_dict(ck['state_dict']); m.to(dev).eval()
    _S.update(model=m, meta=ck['meta'], dev=dev); return _S

def _img(x):
    if isinstance(x, Image.Image): return np.asarray(x.convert('RGB'))
    if isinstance(x, (bytes, bytearray)): return np.asarray(Image.open(io.BytesIO(x)).convert('RGB'))
    if isinstance(x, (str, os.PathLike)): return np.asarray(Image.open(x).convert('RGB'))
    return np.asarray(x)[..., :3].astype(np.uint8)

@torch.no_grad()
def predict_masks(pre):
    S = load(); a = _img(pre); m = S['meta']
    t = torch.from_numpy(((a / 255.0 - np.array(m['mean'])) / np.array(m['std'])).transpose(2, 0, 1)).float()[None]
    H, W = t.shape[-2:]; ph, pw = (32 - H % 32) % 32, (32 - W % 32) % 32
    if ph or pw: t = F.pad(t, (0, pw, 0, ph), mode='reflect')
    p = torch.sigmoid(S['model'](t.to(S['dev'])).float())[0, :, :H, :W].cpu().numpy()
    return p[0], p[1]

def predict_buildings(pre):
    S = load(); m = S['meta']; pm, pb = predict_masks(pre)
    fg = pm > m['thr_mask']; seeds, n = ndi.label(fg & (pb < m['thr_border']))
    if n:
        sizes = ndi.sum(np.ones_like(seeds), seeds, index=np.arange(1, n + 1))
        seeds[np.isin(seeds, np.where(sizes < 4)[0] + 1)] = 0
    inst = watershed(-pm, seeds, mask=fg) if (_SK and seeds.any()) else ndi.label(fg)[0]
    out = []
    for k, sl in enumerate(ndi.find_objects(inst), 1):
        if sl is None: continue
        sub = inst[sl] == k
        if sub.sum() < m['min_area']: continue
        ys, xs = sl; poly = None
        if _SK:
            cs = find_contours(np.pad(sub, 1).astype(float), 0.5)
            if cs:
                c = approximate_polygon(max(cs, key=len), tolerance=1.0)
                poly = [[float(x - 1 + xs.start), float(y - 1 + ys.start)] for y, x in c]
        if not poly or len(poly) < 3:
            poly = [[xs.start, ys.start], [xs.stop, ys.start], [xs.stop, ys.stop], [xs.start, ys.stop], [xs.start, ys.start]]
        out.append({'id': len(out) + 1, 'bbox': [float(xs.start), float(ys.start), float(xs.stop), float(ys.stop)],
                    'area': int(sub.sum()), 'polygon': poly})
    return out
'''
with open(os.path.join(B, "localization_inference.py"), "w") as fh: fh.write(LOC_PY)
jdump({'test_metrics': LOC_TEST, 'end_to_end': E2E, 'hyperparameters': HYPER,
       'note': 'input: PRE-disaster tile; output buildings in image pixels'}, os.path.join(B, "model_card.json"))

# check the bundle reproduces the notebook on one test tile
spec = importlib.util.spec_from_file_location("loc_inf", os.path.join(B, "localization_inference.py"))
LI = importlib.util.module_from_spec(spec); spec.loader.exec_module(LI)
_t = test.iloc[0]
_nb = len(PRED[_t['tile']]['buildings']); _bd = len(LI.predict_buildings(_t['pre_path']))
print(f"bundle check on {_t['tile']}: notebook {_nb} buildings, bundle {_bd} buildings")
assert abs(_nb - _bd) <= max(2, 0.05 * _nb), "bundle disagrees with notebook"

---
## 10. Summary, manifest and ZIP

In [ ]:
print("=" * 64); print("LOCALIZATION - SUMMARY"); print("=" * 64)
print("validation (model selection): best epoch", TRAIN_INFO['best_epoch'], "| val pixel F1", round(TRAIN_INFO['best_val_pixel_f1'], 4))
print("test:", json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in LOC_TEST.items()}))
if E2E: print("end-to-end:", json.dumps({k: v for k, v in E2E.items() if k.startswith(('oracle', 'predicted', 'end_to_end'))}, default=lambda x: round(x, 4)))
jdump({'python': sys.version, 'platform': platform.platform(), 'torch': torch.__version__, 'timm': timm.__version__,
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'skimage': HAVE_SKIMAGE},
      os.path.join(D['repro'], "environment.json"))
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
ZIP = f"{OUT.rstrip('/')}_{stamp}.zip"
with zipfile.ZipFile(ZIP, "w", allowZip64=True) as zf:
    for root, _, fs in os.walk(OUT):
        for fn in sorted(fs):
            full = os.path.join(root, fn)
            zf.write(full, os.path.join("localization_results", os.path.relpath(full, OUT)),
                     compress_type=zipfile.ZIP_STORED if fn.endswith(('.pth', '.png')) else zipfile.ZIP_DEFLATED)
with zipfile.ZipFile(ZIP) as zf: assert zf.testzip() is None
print(f"ZIP OK: {ZIP} ({os.path.getsize(ZIP)/1e6:.0f} MB)")